In [ ]:
# IMPORTS
import random
import sys
import warnings
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bayes_opt import BayesianOptimization
from bayes_opt.event import Events  # bayes_opt<3 API
from bayes_opt.logger import JSONLogger  # bayes_opt<3 API
from river import anomaly, preprocessing, utils
from river.metrics import F1
from sklearn.decomposition import PCA

sys.path.insert(1, str(Path.cwd().parent))
from functions.anomaly import ConditionalGaussianScorer
from functions.compose import build_model, convert_to_nested_dict
from functions.evaluate import (
    build_fit_evaluate,
    print_stats,
    progressive_val_predict,
)
from functions.proba import MultivariateGaussian
from functions.reunanen import ReunanenScorer

# CONSTANTS
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)


# FUNCTIONS
def get_random_samples(
    df: pd.DataFrame,
    num_samples: int = 10000,
) -> pd.DataFrame:
    """Return df unchanged or a random subsample of num_samples rows."""
    if len(df) <= num_samples:
        return df
    return df.sample(n=num_samples, random_state=RANDOM_STATE)


def plot_detection(df: pd.DataFrame, y_pred: list[int] | pd.Series) -> None:
    """Plot true anomalies and predictions in 2-D PCA space."""
    df["pred"] = y_pred
    if "anomaly" in df.columns:
        df = get_random_samples(df)
        if len(df.columns) >= 4:
            # Separate the feature columns from the target column ("anomaly")
            X = df.drop(columns=["anomaly", "pred"])
            y = df["anomaly"]
            y_pred = df["pred"]

            # Apply PCA to reduce the feature columns to 2 components
            pca = PCA(n_components=2)
            X_pca = pca.fit_transform(X)

            # Create a new DataFrame with reduced components and
            # "anomaly" column
            df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
            df_pca["anomaly"] = y.to_numpy()
            df_pca["pred"] = y_pred.to_numpy()
        else:
            df_pca = pd.DataFrame(df.reset_index().copy())
            df_pca.columns = ["PC1", "PC2", "anomaly", "pred"]

        # Plot the 2D scatter plot
        plt.scatter(
            df_pca[df_pca["anomaly"] == 0]["PC1"],
            df_pca[df_pca["anomaly"] == 0]["PC2"],
        )
        plt.scatter(
            df_pca[df_pca["anomaly"] == 1]["PC1"],
            df_pca[df_pca["anomaly"] == 1]["PC2"],
            facecolors="none",
            edgecolors="r",
            linewidths=0.5,
        )
        plt.scatter(
            df_pca[df_pca["pred"] == 1]["PC1"],
            df_pca[df_pca["pred"] == 1]["PC2"],
            marker="x",
            linewidths=1,
        )
        plt.xticks(())
        plt.yticks(())


def save_results_y(df_ys: pd.DataFrame, path: str) -> None:
    """Save prediction DataFrame to a CSV file inside the given directory."""
    Path(path).mkdir(parents=True, exist_ok=True)
    df_ys.to_csv(f"{path}/ys.csv", index=False)


def save_results_metrics(metrics_res: pd.DataFrame, path: str) -> None:
    """Save metrics DataFrame to a CSV file inside the given directory."""
    Path(path).mkdir(parents=True, exist_ok=True)
    metrics_res.to_csv(f"{path}/metrics.csv")


# MODS
class QuantileFilter(anomaly.QuantileFilter):
    """QuantileFilter subclass delegating classification to self.classify."""

    def __init__(
        self,
        anomaly_detector: anomaly.base.AnomalyDetector,
        q: float,
        protect_anomaly_detector: bool = True,
    ) -> None:
        """Initialise with anomaly detector, quantile, and protection flag."""
        super().__init__(
            anomaly_detector=anomaly_detector,
            protect_anomaly_detector=protect_anomaly_detector,
            q=q,
        )

    def predict_one(self, *args: dict) -> bool:
        """Score the sample and classify it via the quantile threshold."""
        score = self.score_one(*args)
        return self.classify(score)


# SETTINGS

# DETECTION ALGORITHMS
detection_algorithms = [
    (
        "Conditional Gaussian Scorer",
        [[ConditionalGaussianScorer, [utils.Rolling, MultivariateGaussian]]],
        {
            "ConditionalGaussianScorer__threshold": (0.65, 0.99994),
            "Rolling__window_size__round": (150, 10000),
            "ConditionalGaussianScorer__t_a__int": (50, 2000),
            "ConditionalGaussianScorer__grace_period__round": (50, 1000),
        },
    ),
    (
        "Half-Space Trees",
        [preprocessing.MinMaxScaler, [QuantileFilter, anomaly.HalfSpaceTrees]],
        {
            "QuantileFilter__q": (0.65, 0.99994),
            "HalfSpaceTrees__n_trees__round": (1, 20),
            "HalfSpaceTrees__height__round": (1, 16),
            "HalfSpaceTrees__window_size__round": (100, 400),
        },
    ),
    (
        "One-Class SVM",
        [preprocessing.StandardScaler, [QuantileFilter, anomaly.OneClassSVM]],
        {
            "QuantileFilter__q": (0.65, 0.99994),
            "OneClassSVM__intercept_lr": (0.005, 0.02),
        },
    ),
    (
        # Reunanen et al. (2020) streaming autoencoder detector: a tuned
        # like-for-like online baseline (same bayes_opt budget as AID),
        # not a strawman. It self-thresholds, so no QuantileFilter wrapper.
        "Reunanen Scorer",
        [ReunanenScorer],
        {
            "ReunanenScorer__n_hidden__round": (2, 10),
            "ReunanenScorer__lr": (0.02, 0.3),
            "ReunanenScorer__k": (2.0, 4.0),
            "ReunanenScorer__M__round": (500, 5000),
        },
    ),
]

# DATASETS
datasets = [
    {
        "name": "SKAB",
        "data": pd.read_csv(
            "data/multivariate/alldata_skab.csv",
            index_col=0,
        ).dropna(axis=0),
        "anomaly_col": "is_anomaly",
        "drop": "changepoint",
    },
]

# PLOT CONFIG
plt.figure(figsize=(len(detection_algorithms) * 2 + 4, 12.5))
plt.subplots_adjust(
    left=0.02,
    right=0.98,
    bottom=0.001,
    top=0.96,
    wspace=0.05,
    hspace=0.01,
)
plot_num = 1

# RUN
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for dataset in datasets:
        # PREPROCESS DATA
        df = dataset["data"]
        assert isinstance(df, pd.DataFrame)
        df.index = pd.to_timedelta(
            range(len(df)),
            "min",
        ) + pd.Timestamp.utcnow().replace(microsecond=0)
        if isinstance(dataset["anomaly_col"], str):
            df = df.rename(columns={dataset["anomaly_col"]: "anomaly"})
        elif isinstance(dataset["anomaly_col"], pd.Series):
            df_y = dataset["anomaly_col"]
            df["anomaly"] = df_y.rename("anomaly").to_numpy()
        if dataset["drop"] is not None:
            df = df.drop(columns=dataset["drop"])

        df_ys = df[["anomaly"]].copy()
        # RUN EACH MODEL AGAINST DATASET
        for alg in detection_algorithms:
            # INITIALIZE OPTIMIZER
            pbounds = alg[2]
            mod_fun = partial(
                build_fit_evaluate,
                alg[1],
                df,
                F1(),
                map_cluster_to_rc=False,
                drop_no_support=False,
            )

            # TUNE HYPERPARAMETERS
            optimizer = BayesianOptimization(
                f=mod_fun,
                pbounds=pbounds,
                verbose=2,
                random_state=RANDOM_STATE,
                allow_duplicate_points=True,
            )
            logger = JSONLogger(
                path=f"./.results/{dataset['name']}-{alg[0]}.log",
            )
            # subscribe() removed in bayes_opt 3.x; study ran on 2.x.
            optimizer.subscribe(
                Events.OPTIMIZATION_END,
                logger,
            )
            optimizer.maximize(init_points=20, n_iter=100)
            opt_max = optimizer.max
            assert opt_max is not None
            params = convert_to_nested_dict(opt_max["params"])
            model = build_model(alg[1], params)
            if hasattr(model, "seed"):
                model.seed = RANDOM_STATE
            if hasattr(model, "random_state"):
                model.random_state = RANDOM_STATE
            # USE TUNED MODEL
            # PROGRESSIVE PREDICT
            y_pred, _ = progressive_val_predict(model, df, metrics=None)

            # SAVE PREDICITONS
            df_ys[f"{alg[0]}__{params}"] = y_pred

            #  PRINT OUT LAST DETECTION RESULTS
            print_stats(df, y_pred)
            plt.subplot(len(datasets), len(detection_algorithms), plot_num)
            plot_detection(df, y_pred)
            plot_num += 1

        # LOAD RESULTS
        #  Save
        dir_path = f".results/{dataset['name']}"
        save_results_y(df_ys, f".results/{dataset['name']}")

plt.show()

In [20]:
import sys
from pathlib import Path

from river.metrics import (
    F1,
    ROCAUC,
    ClassificationReport,
    Precision,
    Recall,
    RollingROCAUC,
)
from river.stats import Mean

sys.path.insert(1, str(Path.cwd().parent))
from functions.evaluate import batch_save_evaluate_metrics


class MeanRollingROCAUC(RollingROCAUC):
    """Rolling ROCAUC variant that returns the mean over the rolling window."""

    def __init__(self, window_size: int = 1000, pos_val: bool = True) -> None:
        """Initialise with rolling window size and positive class value."""
        super().__init__(window_size, pos_val)
        self.mean = Mean()

    def update(
        self,
        y_true: bool,
        y_pred: bool | float | dict[bool, float],
    ) -> None:
        """Update the rolling ROCAUC and accumulate the running mean."""
        super().update(y_true, y_pred)
        self.mean.update(super().get())

    def get(self) -> float:
        """Return the mean of all rolling ROCAUC values seen so far."""
        return self.mean.get()


metrics = [
    Precision(),
    Recall(),
    F1(),
    ROCAUC(),
    MeanRollingROCAUC(),
    ClassificationReport(),
]

path = ".results/"

batch_save_evaluate_metrics(metrics, path, task="classification")